# N-Lens System Inversion

This notebook fits lens column parameters using the Glaser bell model physics:

1. **Focal length**: $\displaystyle \frac{1}{f_i} = C_{f,i} \cdot (I_{0,i} + I_{0,i}w)^2$
2. **Image rotation**: $\displaystyle \psi_i = K_v \cdot (I_{0,i} + I_{0,i}w)$

where wobble $w$ is a relative current perturbation applied to one lens at a time.

## Fit parameters

| Symbol | Meaning | Per-lens? |
|--------|---------|-----------|
| $d_j$ | Propagation distance | yes ($N+1$ total) |
| $I_{0,i}$ | Nominal excitation (ampere-turns) | yes ($N$ total) |
| $C_{f,i}$ | Lens geometry constant (bore, gap, pole-piece) | yes ($N$ total) |
| $K_v$ | Rotation constant $= e\mu_0/(2m_e v)$ | **no — known from voltage** |

In [9]:
import sys
sys.path.insert(0, '../../src')

import jax
import jax.numpy as jnp
import numpy as np
import sympy as sp
from IPython.display import display
from scipy.optimize import least_squares as scipy_least_squares
import time

from temgym_core.transfer_matrices import propagation_matrix, lens_matrix

jax.config.update("jax_enable_x64", True)

print("Imports successful")

Imports successful


In [10]:
# ================================================================
# PHYSICAL CONSTANTS (SI)
# ================================================================
E_CHARGE = 1.602176634e-19    # elementary charge (C)
M_E      = 9.1093837015e-31   # electron mass (kg)
C_LIGHT  = 299792458.0        # speed of light (m/s)
MU_0     = 1.25663706212e-6   # vacuum permeability (H/m)

# ================================================================
# ACCELERATING VOLTAGE → K_v (KNOWN, NOT FITTED)
# ================================================================
U_ACCEL = 200e3  # 200 kV

gamma_rel  = 1 + E_CHARGE * U_ACCEL / (M_E * C_LIGHT**2)
v_electron = C_LIGHT * np.sqrt(1 - 1 / gamma_rel**2)
K_ROT      = E_CHARGE * MU_0 / (2 * M_E * v_electron)   # rad / AT

print(f"Physical constants loaded:")
print(f"  Accelerating voltage: U  = {U_ACCEL/1e3:.0f} kV")
print(f"  Rotation constant:    K_rot = {K_ROT:.6e} rad/AT")

Physical constants loaded:
  Accelerating voltage: U  = 200 kV
  Rotation constant:    K_rot = 5.301506e-04 rad/AT


## Forward Model: ABCD + Rotation

Compute $A,B$ from a chain of propagation and thin-lens matrices, and compute rotation from the $\theta(I)$ model.

In [11]:
def f_from_excitation(Cf_i, I0_i, w):
    """Focal length from Glaser Eq. 16: 1/f = Cf * (I0*(1+w))^2."""
    return 1.0 / (Cf_i * (I0_i * (1.0 + w)) ** 2)


def psi_from_excitation(Kv, I0_i, w):
    """Image rotation from Glaser Eq. 17: ψ = Kv * I0 * (1+w)."""
    return Kv * I0_i * (1.0 + w)


def build_abcd(dists, focals, xp=jnp):
    """Construct ABCD matrix for N-lens system.

    dists: length N+1, focals: length N
    M = P(d_{N+1}) L_N ... L_1 P(d_1)
    """
    M = propagation_matrix(dists[-1], xp=xp)
    for i in reversed(range(len(focals))):
        M = M @ lens_matrix(focals[i], xp=xp)
        M = M @ propagation_matrix(dists[i], xp=xp)
    return M


def compute_AB(dists, focals):
    M = build_abcd(dists, focals, xp=jnp)
    return M[0, 0], M[0, 1]


def model_measurement(dists, I0, Cf, Kv, wobble_lens, w, defocus):
    """Forward model: A, B, ψ from Glaser bell parametrisation.

    Pure JAX — compiled via vmap + Optimistix.

    Args:
        dists: propagation distances (N+1,)
        I0: per-lens excitation currents (N,)
        Cf: per-lens geometry constants (N,)
        Kv: rotation constant (scalar, known from voltage)
        wobble_lens: which lens index to wobble
        w: wobble value (scalar)
        defocus: sample/input defocus (scalar)
    """
    # Focal lengths
    f_use = []
    for i in range(len(I0)):
        w_i = jnp.where(i == wobble_lens, w, 0.0)
        f_use.append(f_from_excitation(Cf[i], I0[i], w_i))
    f_use = jnp.array(f_use)

    # Distances (add defocus to input/first segment)
    d_use = jnp.array(dists)
    d_use = d_use.at[0].add(defocus)

    A, B = compute_AB(d_use, f_use)

    # Total rotation (sum over all lenses)
    psi_total = 0.0
    for i in range(len(I0)):
        w_i = jnp.where(i == wobble_lens, w, 0.0)
        psi_total = psi_total + psi_from_excitation(Kv, I0[i], w_i)

    return A, B, psi_total


## Sympy 5×5 Matrix (Glaser Physics)

Symbolic 5×5 transfer matrix using:

- $1/f_i = C_{f,i} \cdot I_i^2$ (Eq. 16)
- $\psi_i = K_v \cdot I_i$ (Eq. 17)
- Wobble: $I_i = I_{0,i} + I_{0,i} * w_i$

In [12]:
# Sympy definitions — Glaser parametrisation
Cf1_s, Cf2_s = sp.symbols("C_{f1} C_{f2}", positive=True)
Kv_s = sp.symbols("K_v", positive=True)
I1_0, I2_0 = sp.symbols("I_{01} I_{02}", positive=True)
w1, w2 = sp.symbols("w_1 w_2")
d1, d2, d3 = sp.symbols("d_1 d_2 d_3", positive=True)

# Currents with wobble
I1 = I1_0 * (1 + w1)
I2 = I2_0 * (1 + w2)

# Focal lengths: 1/f = Cf * I^2  (Eq. 16)
f1 = 1 / (Cf1_s * I1**2)
f2 = 1 / (Cf2_s * I2**2)

# Rotations: ψ = Kv * I  (Eq. 17)
psi1 = Kv_s * I1
psi2 = Kv_s * I2


def propagation_5x5(d):
    return sp.Matrix([
        [1, 0, d, 0, 0],
        [0, 1, 0, d, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 1],
    ])


def lens_5x5(f):
    return sp.Matrix([
        [1, 0, 0, 0, 0],
        [0, 1, 0, 0, 0],
        [-1 / f, 0, 1, 0, 0],
        [0, -1 / f, 0, 1, 0],
        [0, 0, 0, 0, 1],
    ])


def rotation_5x5(theta):
    c = sp.cos(theta)
    s = sp.sin(theta)
    return sp.Matrix([
        [c, -s, 0, 0, 0],
        [s,  c, 0, 0, 0],
        [0,  0, c, -s, 0],
        [0,  0, s,  c, 0],
        [0,  0, 0,  0, 1],
    ])


M5 = (
    propagation_5x5(d3)
    * rotation_5x5(psi2)
    * lens_5x5(f2)
    * propagation_5x5(d2)
    * rotation_5x5(psi1)
    * lens_5x5(f1)
    * propagation_5x5(d1)
)

print("Glaser bell model relations:")
display(sp.Eq(sp.Symbol("1/f_1"), 1/f1))
display(sp.Eq(sp.Symbol("1/f_2"), 1/f2))
display(sp.Eq(sp.Symbol("ψ_1"), psi1))
display(sp.Eq(sp.Symbol("ψ_2"), psi2))

print("\nKey: K_v is KNOWN from voltage; C_{f,i} are per-lens geometry constants")
print("\n5×5 transfer matrix for a 2-lens system (propagation + lens + rotation):")
display(M5)

Glaser bell model relations:


Eq(1/f_1, C_{f1}*I_{01}**2*(w_1 + 1)**2)

Eq(1/f_2, C_{f2}*I_{02}**2*(w_2 + 1)**2)

Eq(ψ_1, I_{01}*K_v*(w_1 + 1))

Eq(ψ_2, I_{02}*K_v*(w_2 + 1))


Key: K_v is KNOWN from voltage; C_{f,i} are per-lens geometry constants

5×5 transfer matrix for a 2-lens system (propagation + lens + rotation):


Matrix([
[  -C_{f1}*I_{01}**2*(w_1 + 1)**2*((d_2*(C_{f2}*I_{02}**2*d_3*(w_2 + 1)**2*sin(I_{02}*K_v*(w_2 + 1)) - sin(I_{02}*K_v*(w_2 + 1))) - d_3*sin(I_{02}*K_v*(w_2 + 1)))*sin(I_{01}*K_v*(w_1 + 1)) + (d_2*(-C_{f2}*I_{02}**2*d_3*(w_2 + 1)**2*cos(I_{02}*K_v*(w_2 + 1)) + cos(I_{02}*K_v*(w_2 + 1))) + d_3*cos(I_{02}*K_v*(w_2 + 1)))*cos(I_{01}*K_v*(w_1 + 1))) + (C_{f2}*I_{02}**2*d_3*(w_2 + 1)**2*sin(I_{02}*K_v*(w_2 + 1)) - sin(I_{02}*K_v*(w_2 + 1)))*sin(I_{01}*K_v*(w_1 + 1)) + (-C_{f2}*I_{02}**2*d_3*(w_2 + 1)**2*cos(I_{02}*K_v*(w_2 + 1)) + cos(I_{02}*K_v*(w_2 + 1)))*cos(I_{01}*K_v*(w_1 + 1)),    -C_{f1}*I_{01}**2*(w_1 + 1)**2*((d_2*(C_{f2}*I_{02}**2*d_3*(w_2 + 1)**2*sin(I_{02}*K_v*(w_2 + 1)) - sin(I_{02}*K_v*(w_2 + 1))) - d_3*sin(I_{02}*K_v*(w_2 + 1)))*cos(I_{01}*K_v*(w_1 + 1)) - (d_2*(-C_{f2}*I_{02}**2*d_3*(w_2 + 1)**2*cos(I_{02}*K_v*(w_2 + 1)) + cos(I_{02}*K_v*(w_2 + 1))) + d_3*cos(I_{02}*K_v*(w_2 + 1)))*sin(I_{01}*K_v*(w_1 + 1))) + (C_{f2}*I_{02}**2*d_3*(w_2 + 1)**2*sin(I_{02}*K_v*(w_2 + 

## Generate Synthetic Measurements

Compute synthetic $A,B,\theta$ for each wobble setting (one lens at a time).

In [13]:
def generate_measurements(d_true, I0_true, Cf_true, Kv, wobble_values, defocus_values):
    """Generate synthetic measurements as vectorized JAX arrays.

    Returns dict: {wobble_lens, wobble, defocus, A, B, psi}
    """
    wl_list, w_list, df_list = [], [], []
    A_list, B_list, psi_list = [], [], []

    for lens_idx in range(len(I0_true)):
        for w in wobble_values:
            for defocus in defocus_values:
                A, B, psi = model_measurement(
                    d_true, I0_true, Cf_true, Kv, lens_idx, w, defocus
                )
                wl_list.append(lens_idx)
                w_list.append(float(w))
                df_list.append(float(defocus))
                A_list.append(float(A))
                B_list.append(float(B))
                psi_list.append(float(psi))

    return {
        "wobble_lens": jnp.array(wl_list, dtype=jnp.int32),
        "wobble": jnp.array(w_list),
        "defocus": jnp.array(df_list),
        "A": jnp.array(A_list),
        "B": jnp.array(B_list),
        "psi": jnp.array(psi_list),
    }

## Residuals for Least Squares

Define residuals for $A$, $B$, and $\theta$ and build a loss function.

In [14]:
def make_residual_fn(measurements, n_lenses, scales, Kv):
    """vmap-vectorized residual function (Glaser parametrisation).

    Parameter vector: [d1,...,d_{N+1}, I0_1,...,I0_N, Cf_1,...,Cf_N]
    Total: 3N+1 parameters.

    K_v is KNOWN (not fitted) — eliminates the f ↔ ψ degeneracy.
    """
    meas_wl  = measurements["wobble_lens"]
    meas_w   = measurements["wobble"]
    meas_df  = measurements["defocus"]
    meas_A   = measurements["A"]
    meas_B   = measurements["B"]
    meas_psi = measurements["psi"]

    A_scale, B_scale, psi_scale = scales
    n_dist = n_lenses + 1
    n_I0   = n_lenses

    @jax.jit
    def residual_fn(params):
        d  = params[:n_dist]
        I0 = params[n_dist : n_dist + n_I0]
        Cf = params[n_dist + n_I0 : n_dist + 2 * n_I0]

        def single_pred(wl, w, df):
            return model_measurement(d, I0, Cf, Kv, wl, w, df)

        A_pred, B_pred, psi_pred = jax.vmap(single_pred)(meas_wl, meas_w, meas_df)

        res_A   = (A_pred   - meas_A)   / A_scale
        res_B   = (B_pred   - meas_B)   / B_scale
        res_psi = (psi_pred - meas_psi) / psi_scale

        return jnp.concatenate([res_A, res_B, res_psi])

    return residual_fn


def make_loss_fn(measurements, n_lenses, scales, Kv):
    """Loss = sum of squared residuals."""
    residual_fn = make_residual_fn(measurements, n_lenses, scales, Kv)

    def loss_fn(params):
        r = residual_fn(params)
        return jnp.sum(r ** 2)

    return loss_fn


print("Residual and loss functions ready")

Residual and loss functions ready


In [15]:

def make_residual_fn_constrained(measurements, n_lenses, scales, Kv, total_distance=None):
    """vmap-vectorized residual function with optional total distance constraint.

    If total_distance is specified:
        Parameter vector: [d1,...,d_N, I0_1,...,I0_N, Cf_1,...,Cf_N]
        where d_{N+1} = total_distance - sum(d_1...d_N)
        Total: 3N parameters (one less distance)
    
    If total_distance is None:
        Parameter vector: [d1,...,d_{N+1}, I0_1,...,I0_N, Cf_1,...,Cf_N]
        Total: 3N+1 parameters (standard)

    K_v is KNOWN (not fitted) — eliminates the f ↔ ψ degeneracy.
    """
    meas_wl  = measurements["wobble_lens"]
    meas_w   = measurements["wobble"]
    meas_df  = measurements["defocus"]
    meas_A   = measurements["A"]
    meas_B   = measurements["B"]
    meas_psi = measurements["psi"]

    A_scale, B_scale, psi_scale = scales
    n_I0   = n_lenses
    
    if total_distance is not None:
        n_dist_fit = n_lenses  # Only fit N distances, derive the (N+1)th
    else:
        n_dist_fit = n_lenses + 1  # Fit all N+1 distances

    @jax.jit
    def residual_fn(params):
        if total_distance is not None:
            # Reconstruct full distance array from first N + constraint
            d_fit = params[:n_dist_fit]
            d_last = total_distance - jnp.sum(d_fit)
            d = jnp.concatenate([d_fit, jnp.array([d_last])])
        else:
            d = params[:n_dist_fit]
        
        I0 = params[n_dist_fit : n_dist_fit + n_I0]
        Cf = params[n_dist_fit + n_I0 : n_dist_fit + 2 * n_I0]

        def single_pred(wl, w, df):
            return model_measurement(d, I0, Cf, Kv, wl, w, df)

        A_pred, B_pred, psi_pred = jax.vmap(single_pred)(meas_wl, meas_w, meas_df)

        res_A   = (A_pred   - meas_A)   / A_scale
        res_B   = (B_pred   - meas_B)   / B_scale
        res_psi = (psi_pred - meas_psi) / psi_scale

        return jnp.concatenate([res_A, res_B, res_psi])

    return residual_fn


## System Configuration & Fitting

Define your system parameters in the cell below and run the fitting routine. The `setup_system()` function initializes the system configuration from your chosen parameters, and `fit_system()` executes the complete optimization pipeline.


In [35]:
def setup_system(N_LENSES_FIT, D_TRUE_FIT, I0_TRUE_FIT, CF_TRUE_FIT, 
                 wobble_grids_fit, defocus_grids_fit, TOTAL_DISTANCE_KNOWN=None, rng_seed=None):
    """
    Setup system configuration from user parameters.
    
    Returns dict with all necessary parameters for fitting.
    """
    # Initialize constraint
    if TOTAL_DISTANCE_KNOWN == True:
        TOTAL_DISTANCE_FIT = D_TRUE_FIT.sum()
    else:
        TOTAL_DISTANCE_FIT = None
    
    # Generate RNG-based initial guesses
    if rng_seed is None:
        rng_seed = {2: 90, 3: 43, 4: 44, 5: 45}.get(N_LENSES_FIT, 90)
    
    rng = np.random.default_rng(rng_seed)
    n_dist = N_LENSES_FIT + 1
    n_I0 = N_LENSES_FIT
    n_Cf = N_LENSES_FIT
    
    # Simple random initialization within broad, reasonable ranges
    if TOTAL_DISTANCE_FIT is not None:
        d_raw = rng.random(n_dist) + 1e-6
        x0_dist_fit = TOTAL_DISTANCE_FIT * d_raw / np.sum(d_raw)
    else:
        x0_dist_fit = rng.uniform(1e-3, 0.3, size=n_dist)       # meters
    x0_I0_fit = rng.uniform(10.0, 1e4, size=n_I0)            # AT
    x0_Cf_fit = 10.0 ** rng.uniform(-6.0, -4.0, size=n_Cf)   # log-uniform
    
    # Compute true parameters
    x_true = np.concatenate([D_TRUE_FIT, I0_TRUE_FIT, CF_TRUE_FIT])
    x0_fit = np.concatenate([x0_dist_fit, x0_I0_fit, x0_Cf_fit])
    
    config = {
        'N_LENSES_FIT': N_LENSES_FIT,
        'D_TRUE_FIT': D_TRUE_FIT,
        'I0_TRUE_FIT': I0_TRUE_FIT,
        'CF_TRUE_FIT': CF_TRUE_FIT,
        'wobble_grids_fit': wobble_grids_fit,
        'defocus_grids_fit': defocus_grids_fit,
        'TOTAL_DISTANCE_FIT': TOTAL_DISTANCE_FIT,
        'x0_dist_fit': x0_dist_fit,
        'x0_I0_fit': x0_I0_fit,
        'x0_Cf_fit': x0_Cf_fit,
        'x_true': x_true,
        'x0_fit': x0_fit,
    }
    
    return config


def fit_system(config, K_ROT, opt_params=None, multistart_runs=10, multistart_seed=0, multistart_enabled=True, refine_runs=3, refine_loss_threshold=1e-6):
    """
    Execute the fitting routine for a configured system.
    
    Parameters:
    -----------
    config : dict
        Output from setup_system()
    K_ROT : float
        Rotation constant (from physical constants)
    opt_params : dict, optional
        scipy.optimize.least_squares parameters (ftol, xtol, gtol, max_nfev)
    multistart_runs : int, optional
        Number of multistart runs (ignored if multistart_enabled is False)
    multistart_seed : int, optional
        Base seed for multistart initial guesses
    multistart_enabled : bool, optional
        If True, run multiple starts and keep the best result
    refine_runs : int, optional
        Number of refinement passes if loss exceeds refine_loss_threshold
    refine_loss_threshold : float, optional
        Run refinement if best loss is above this threshold
    
    Returns:
    --------
    dict with results: x_fit_full, errs_final, convergence table
    """
    if opt_params is None:
        opt_params = {'ftol': 1e-10, 'xtol': 1e-10, 'gtol': 1e-10, 'max_nfev': 15000}
    
    # Unpack configuration
    N_LENSES_FIT = config['N_LENSES_FIT']
    D_TRUE_FIT = config['D_TRUE_FIT']
    I0_TRUE_FIT = config['I0_TRUE_FIT']
    CF_TRUE_FIT = config['CF_TRUE_FIT']
    wobble_grids_fit = config['wobble_grids_fit']
    defocus_grids_fit = config['defocus_grids_fit']
    TOTAL_DISTANCE_FIT = config['TOTAL_DISTANCE_FIT']
    x0_dist_fit = config['x0_dist_fit']
    x0_I0_fit = config['x0_I0_fit']
    x0_Cf_fit = config['x0_Cf_fit']
    x_true = config['x_true']
    x0_fit = config['x0_fit']
    
    # Print system summary
    print("=" * 70)
    print(f"FITTING {N_LENSES_FIT}-LENS SYSTEM (Glaser Parametrisation)")
    print("=" * 70)
    print(f"Distances: {D_TRUE_FIT*1e3} mm")
    for i in range(N_LENSES_FIT):
        psi_i = K_ROT * I0_TRUE_FIT[i]
        print(f"  Lens {i+1}:  I0={I0_TRUE_FIT[i]:.0f} AT,  Cf={CF_TRUE_FIT[i]:.3e}"
              f"  →  f={1.0/(CF_TRUE_FIT[i]*I0_TRUE_FIT[i]**2)*1e3:.2f} mm,  ψ={np.degrees(psi_i):.1f}°")
    
    print(f"\nParameters: {len(x_true)}  ({N_LENSES_FIT + 1} dist + {N_LENSES_FIT} I0 + {N_LENSES_FIT} Cf)")
    if TOTAL_DISTANCE_FIT is not None:
        print(f"\n>>> Using CONSTRAINT: Known total distance = {TOTAL_DISTANCE_FIT*1000:.1f} mm")
        print(f"    Parameters reduced to {3*N_LENSES_FIT}  ({N_LENSES_FIT} dist + {N_LENSES_FIT} I0 + {N_LENSES_FIT} Cf)")
    else:
        print(f"\n>>> NO CONSTRAINT: Fitting all {len(x_true)} parameters")
    
    start_count = multistart_runs if multistart_enabled else 1
    total_distance_known = TOTAL_DISTANCE_FIT is not None
    print(f"\nMultistart runs: {start_count} (base seed={multistart_seed})")
    
    best_loss = np.inf
    best_err = np.inf
    best_start = None
    best_x_fit_params_full = None
    best_errs_final = None
    
    for start_idx in range(start_count):
        if multistart_enabled:
            seed = multistart_seed + start_idx
            start_config = setup_system(
                N_LENSES_FIT, D_TRUE_FIT, I0_TRUE_FIT, CF_TRUE_FIT,
                wobble_grids_fit, defocus_grids_fit, total_distance_known, rng_seed=seed
            )
            x0_dist_fit = start_config['x0_dist_fit']
            x0_I0_fit = start_config['x0_I0_fit']
            x0_Cf_fit = start_config['x0_Cf_fit']
            x0_fit = start_config['x0_fit']
            print(f"\nStart {start_idx + 1}/{start_count} (seed={seed})")
        else:
            print(f"\nStart {start_idx + 1}/{start_count}")
        
        print(f"Initial guess (perturbed):")
        print(f"  d:  {x0_dist_fit*1e3} mm")
        print(f"  I0: {x0_I0_fit} AT")
        print(f"  Cf: {x0_Cf_fit}")
        
        print(f"\n{'Wobble':>8s} {'Defocus':>8s} {'Meas':>5s} {'Max Err':>10s} {'Loss':>12s} {'Time':>7s}")
        print("-" * 60)
        
        # Run optimization over all wobble/defocus configurations
        x_fit_params = None
        last_loss = None
        last_errs = None
        for wn, ws in wobble_grids_fit.items():
            for dn, ds in defocus_grids_fit.items():
                # Generate measurements
                m = generate_measurements(D_TRUE_FIT, I0_TRUE_FIT, CF_TRUE_FIT, K_ROT, ws, ds)
                sc = (
                    max(np.max(np.abs(m["A"])),   1e-12),
                    max(np.max(np.abs(m["B"])),   1e-12),
                    max(np.max(np.abs(m["psi"])), 1e-12),
                )
                
                # Choose residual function
                if TOTAL_DISTANCE_FIT is not None:
                    rfn = make_residual_fn_constrained(m, N_LENSES_FIT, sc, K_ROT, total_distance=TOTAL_DISTANCE_FIT)
                    x0_use = np.concatenate([x0_dist_fit[:N_LENSES_FIT], x0_I0_fit, x0_Cf_fit])
                    n_dist_use = N_LENSES_FIT
                else:
                    rfn = make_residual_fn(m, N_LENSES_FIT, sc, K_ROT)
                    x0_use = x0_fit
                    n_dist_use = N_LENSES_FIT + 1
                
                # Define bounds
                lower_bounds_fit = np.concatenate([
                    np.full(n_dist_use, 1e-3),      # min distance 1 mm
                    np.full(N_LENSES_FIT, 10.0),    # min current 10 AT
                    np.full(N_LENSES_FIT, 1e-6),    # min Cf
                ])
                upper_bounds_fit = np.concatenate([
                    np.full(n_dist_use, 0.5),       # max distance 500 mm
                    np.full(N_LENSES_FIT, 30000.0), # max current 30k AT
                    np.full(N_LENSES_FIT, 1e-4),    # max Cf
                ])
                
                # Optimize
                t0 = time.time()
                sol = scipy_least_squares(
                    fun=rfn,
                    x0=x0_use,
                    bounds=(lower_bounds_fit, upper_bounds_fit),
                    method='trf',
                    **opt_params,
                    verbose=0,
                )
                elapsed = time.time() - t0
                
                x_fit_params = np.array(sol.x)
                r = np.array(rfn(sol.x))
                
                # Reconstruct full parameter vector if constrained
                if TOTAL_DISTANCE_FIT is not None:
                    x_fit_params_full = np.concatenate([
                        x_fit_params[:N_LENSES_FIT], 
                        np.array([TOTAL_DISTANCE_FIT - np.sum(x_fit_params[:N_LENSES_FIT])]), 
                        x_fit_params[N_LENSES_FIT:]
                    ])
                    errs = np.abs((x_fit_params_full - x_true) / (x_true + 1e-30)) * 100
                else:
                    x_fit_params_full = x_fit_params
                    errs = np.abs((x_fit_params - x_true) / (x_true + 1e-30)) * 100
                
                loss = np.sum(r**2)
                n_meas = len(ws) * len(ds) * N_LENSES_FIT
                s = "✓" if np.max(errs) < 1.0 else ("△" if np.max(errs) < 50. else "✗")
                print(f"{wn:>8s} {dn:>8s} {n_meas:>5d}   {np.max(errs):>7.1f}% {s} {loss:>12.3e} {elapsed:>6.2f}s")
                last_loss = loss
                last_errs = errs
        
        if x_fit_params is None:
            continue
        
        # Keep the best run by loss, with max-error as tiebreak
        last_err_max = float(np.max(last_errs)) if last_errs is not None else np.inf
        if (last_loss is not None) and (last_loss < best_loss or (np.isclose(last_loss, best_loss) and last_err_max < best_err)):
            best_loss = last_loss
            best_err = last_err_max
            best_start = start_idx + 1
            best_x_fit_params_full = x_fit_params_full
            best_errs_final = last_errs
    
    if best_x_fit_params_full is None:
        raise RuntimeError("Multistart failed to produce a solution.")
    
    x_fit_params_full = best_x_fit_params_full
    errs_final = best_errs_final
    print(f"\nBest run: {best_start}/{start_count}  loss={best_loss:.3e}  max_err={best_err:.2f}%")
    
    if best_loss > refine_loss_threshold and refine_runs > 0:
        print(f"\nRefinement: {refine_runs} pass(es) (threshold={refine_loss_threshold:.3e})")
        
        # Seed refinement from the current best solution
        x0_dist_fit = best_x_fit_params_full[:N_LENSES_FIT + 1]
        x0_I0_fit = best_x_fit_params_full[N_LENSES_FIT + 1 : N_LENSES_FIT + 1 + N_LENSES_FIT]
        x0_Cf_fit = best_x_fit_params_full[N_LENSES_FIT + 1 + N_LENSES_FIT : N_LENSES_FIT + 1 + 2 * N_LENSES_FIT]
        x0_fit = best_x_fit_params_full
        
        for refine_idx in range(refine_runs):
            print(f"\nRefine pass {refine_idx + 1}/{refine_runs}")
            print(f"\n{'Wobble':>8s} {'Defocus':>8s} {'Meas':>5s} {'Max Err':>10s} {'Loss':>12s} {'Time':>7s}")
            print("-" * 60)
            
            x_fit_params = None
            last_loss = None
            last_errs = None
            for wn, ws in wobble_grids_fit.items():
                for dn, ds in defocus_grids_fit.items():
                    m = generate_measurements(D_TRUE_FIT, I0_TRUE_FIT, CF_TRUE_FIT, K_ROT, ws, ds)
                    sc = (
                        max(np.max(np.abs(m["A"])),   1e-12),
                        max(np.max(np.abs(m["B"])),   1e-12),
                        max(np.max(np.abs(m["psi"])), 1e-12),
                    )
                    
                    if TOTAL_DISTANCE_FIT is not None:
                        rfn = make_residual_fn_constrained(m, N_LENSES_FIT, sc, K_ROT, total_distance=TOTAL_DISTANCE_FIT)
                        x0_use = np.concatenate([x0_dist_fit[:N_LENSES_FIT], x0_I0_fit, x0_Cf_fit])
                        n_dist_use = N_LENSES_FIT
                    else:
                        rfn = make_residual_fn(m, N_LENSES_FIT, sc, K_ROT)
                        x0_use = x0_fit
                        n_dist_use = N_LENSES_FIT + 1
                    
                    lower_bounds_fit = np.concatenate([
                        np.full(n_dist_use, 1e-3),
                        np.full(N_LENSES_FIT, 10.0),
                        np.full(N_LENSES_FIT, 1e-6),
                    ])
                    upper_bounds_fit = np.concatenate([
                        np.full(n_dist_use, 0.5),
                        np.full(N_LENSES_FIT, 30000.0),
                        np.full(N_LENSES_FIT, 1e-4),
                    ])
                    
                    t0 = time.time()
                    sol = scipy_least_squares(
                        fun=rfn,
                        x0=x0_use,
                        bounds=(lower_bounds_fit, upper_bounds_fit),
                        method='trf',
                        **opt_params,
                        verbose=0,
                    )
                    elapsed = time.time() - t0
                    
                    x_fit_params = np.array(sol.x)
                    r = np.array(rfn(sol.x))
                    
                    if TOTAL_DISTANCE_FIT is not None:
                        x_fit_params_full = np.concatenate([
                            x_fit_params[:N_LENSES_FIT],
                            np.array([TOTAL_DISTANCE_FIT - np.sum(x_fit_params[:N_LENSES_FIT])]),
                            x_fit_params[N_LENSES_FIT:]
                        ])
                        errs = np.abs((x_fit_params_full - x_true) / (x_true + 1e-30)) * 100
                    else:
                        x_fit_params_full = x_fit_params
                        errs = np.abs((x_fit_params - x_true) / (x_true + 1e-30)) * 100
                    
                    loss = np.sum(r**2)
                    n_meas = len(ws) * len(ds) * N_LENSES_FIT
                    s = "✓" if np.max(errs) < 1.0 else ("△" if np.max(errs) < 50. else "✗")
                    print(f"{wn:>8s} {dn:>8s} {n_meas:>5d}   {np.max(errs):>7.1f}% {s} {loss:>12.3e} {elapsed:>6.2f}s")
                    last_loss = loss
                    last_errs = errs
            
            if x_fit_params is None:
                continue
            
            last_err_max = float(np.max(last_errs)) if last_errs is not None else np.inf
            if (last_loss is not None) and (last_loss < best_loss or (np.isclose(last_loss, best_loss) and last_err_max < best_err)):
                best_loss = last_loss
                best_err = last_err_max
                best_x_fit_params_full = x_fit_params_full
                best_errs_final = last_errs
                x0_dist_fit = best_x_fit_params_full[:N_LENSES_FIT + 1]
                x0_I0_fit = best_x_fit_params_full[N_LENSES_FIT + 1 : N_LENSES_FIT + 1 + N_LENSES_FIT]
                x0_Cf_fit = best_x_fit_params_full[N_LENSES_FIT + 1 + N_LENSES_FIT : N_LENSES_FIT + 1 + 2 * N_LENSES_FIT]
                x0_fit = best_x_fit_params_full
    
    # Print fitted parameters
    x_fit_params_full = best_x_fit_params_full
    errs_final = np.abs((x_fit_params_full - x_true) / (x_true + 1e-30)) * 100
    print(f"\n{'='*70}")
    print(f"FITTED PARAMETERS (from last configuration)")
    print(f"{'='*70}")
    
    n_dist_full = N_LENSES_FIT + 1
    d_fit = x_fit_params_full[:n_dist_full]
    I0_fit = x_fit_params_full[n_dist_full : n_dist_full + N_LENSES_FIT]
    Cf_fit = x_fit_params_full[n_dist_full + N_LENSES_FIT : n_dist_full + 2*N_LENSES_FIT]
    
    print(f"Distances: {d_fit*1e3} mm")
    for i in range(N_LENSES_FIT):
        psi_i = K_ROT * I0_fit[i]
        f_i = 1.0 / (Cf_fit[i] * I0_fit[i]**2)
        print(f"  Lens {i+1}:  I0={I0_fit[i]:.0f} AT,  Cf={Cf_fit[i]:.3e}"
              f"  →  f={f_i*1e3:.2f} mm,  ψ={np.degrees(psi_i):.1f}°")
    
    print(f"\n{'Param':>8s} {'True':>12s} {'Fitted':>12s} {'Error':>8s}")
    print("-" * 45)
    param_names = (
        [f"d{i+1}" for i in range(n_dist_full)]
        + [f"I0_{i+1}" for i in range(N_LENSES_FIT)]
        + [f"Cf_{i+1}" for i in range(N_LENSES_FIT)]
    )
    
    for name, tv, fv, err in zip(param_names, x_true, x_fit_params_full, errs_final):
        s = "✓" if err < 1.0 else ("△" if err < 5.0 else "✗")
        print(f"{name:>8s} {tv:>12.4e} {fv:>12.4e} {err:>7.2f}% {s}")
    
    return {
        'x_fit_params_full': x_fit_params_full,
        'errs_final': errs_final,
        'd_fit': d_fit,
        'I0_fit': I0_fit,
        'Cf_fit': Cf_fit,
        'best_loss': best_loss,
        'best_max_err': best_err,
        'best_start': best_start,
    }

print("Setup and fit functions ready")


Setup and fit functions ready


In [ ]:
# ================================================================
# USER INPUT: System Parameters (Edit these to change configuration)
# ================================================================

# Choose number of lenses and uncomment desired configuration:
N_LENSES_FIT = 5

# Constraint: set to True to enforce total distance, None otherwise
TOTAL_DISTANCE_KNOWN = True

# Multistart controls
MULTISTART_ENABLED = True
MULTISTART_RUNS = 20
MULTISTART_SEED = 0
REFINE_RUNS = 20
REFINE_LOSS_THRESHOLD = 1e-6

wobble_grid_in = np.linspace(-0.1, 0.1, 9)  # Wobble values to simulate
defocus_grid_in = np.linspace(0, 50e-4, 1)              # Defocus values to simulate

# ================================================================
# LENS CONFIGURATION PRESETS
# ================================================================

if N_LENSES_FIT == 2:
    D_TRUE_FIT  = np.array([2e-3, 40e-3, 200e-3])
    I0_TRUE_FIT = np.array([2000.0, 800.0])
    CF_TRUE_FIT = np.array([8.333e-5, 3.125e-5])
    wobble_grids_fit = {"full": wobble_grid_in}
    defocus_grids_fit = {"minimal": defocus_grid_in}
    opt_params = {'ftol': 1e-11, 'xtol': 1e-11, 'gtol': 1e-11, 'max_nfev': 15000}

elif N_LENSES_FIT == 3:
    D_TRUE_FIT  = np.array([2e-3, 20e-3, 40e-3, 200e-3])
    I0_TRUE_FIT = np.array([2000.0, 1000.0, 600.0])
    CF_TRUE_FIT = np.array([8.333e-5, 5.0e-5, 3.472e-5])
    wobble_grids_fit = {"full": wobble_grid_in}
    defocus_grids_fit = {"minimal": defocus_grid_in}
    opt_params = {'ftol': 1e-11, 'xtol': 1e-11, 'gtol': 1e-11, 'max_nfev': 30000}

elif N_LENSES_FIT == 4:
    D_TRUE_FIT  = np.array([2e-3, 10e-3, 20e-3, 40e-3, 200e-3])
    I0_TRUE_FIT = np.array([2000.0, 1200.0, 800.0, 500.0])
    CF_TRUE_FIT = np.array([8.333e-5, 5.787e-5, 3.906e-5, 2.441e-5])
    wobble_grids_fit = {"full": wobble_grid_in}
    defocus_grids_fit = {"minimal": defocus_grid_in}
    opt_params = {'ftol': 1e-8, 'xtol': 1e-8, 'gtol': 1e-8, 'max_nfev': 10000}

elif N_LENSES_FIT == 5:
    D_TRUE_FIT  = np.array([2e-3, 8e-3, 15e-3, 25e-3, 40e-3, 200e-3])
    I0_TRUE_FIT = np.array([3400.0, 3200.0, 3000.0, 2800.0, 2600.0])  # Currents for mag ~10,000
    CF_TRUE_FIT = np.array([8.333e-5, 5.415e-5, 3.951e-5, 2.755e-5, 2.0e-5])
    wobble_grids_fit = {"full": wobble_grid_in}
    defocus_grids_fit = {"minimal": defocus_grid_in}
    opt_params = {'ftol': 1e-8, 'xtol': 1e-8, 'gtol': 1e-8, 'max_nfev': 10000}

else:
    raise ValueError(f"N_LENSES_FIT = {N_LENSES_FIT} not supported. Choose 2, 3, 4, or 5.")

# Setup the system configuration
config = setup_system(N_LENSES_FIT, D_TRUE_FIT, I0_TRUE_FIT, CF_TRUE_FIT,
                      wobble_grids_fit, defocus_grids_fit, TOTAL_DISTANCE_KNOWN)

# Execute fitting
results = fit_system(
    config,
    K_ROT,
    opt_params,
    multistart_runs=MULTISTART_RUNS,
    multistart_seed=MULTISTART_SEED,
    multistart_enabled=MULTISTART_ENABLED,
    refine_runs=REFINE_RUNS,
    refine_loss_threshold=REFINE_LOSS_THRESHOLD,
)


FITTING 5-LENS SYSTEM (Glaser Parametrisation)
Distances: [  2.   8.  15.  25.  40. 200.] mm
  Lens 1:  I0=3400 AT,  Cf=8.333e-05  →  f=1.04 mm,  ψ=103.3°
  Lens 2:  I0=3200 AT,  Cf=5.415e-05  →  f=1.80 mm,  ψ=97.2°
  Lens 3:  I0=3000 AT,  Cf=3.951e-05  →  f=2.81 mm,  ψ=91.1°
  Lens 4:  I0=2800 AT,  Cf=2.755e-05  →  f=4.63 mm,  ψ=85.1°
  Lens 5:  I0=2600 AT,  Cf=2.000e-05  →  f=7.40 mm,  ψ=79.0°

Parameters: 16  (6 dist + 5 I0 + 5 Cf)

>>> Using CONSTRAINT: Known total distance = 290.0 mm
    Parameters reduced to 15  (5 dist + 5 I0 + 5 Cf)

Multistart runs: 20 (base seed=0)

Start 1/20 (seed=0)
Initial guess (perturbed):
  d:  [68.66165781 29.08187882  4.41686585  1.78171114 87.66691149 98.39097489] mm
  I0: [6070.29139991 7297.67064423 5440.81366474 9351.37351364 8160.37700567] AT
  Cf: [1.01269112e-06 5.18571391e-05 1.16726953e-06 2.87945896e-05
 2.24549060e-06]

  Wobble  Defocus  Meas    Max Err         Loss    Time
------------------------------------------------------------
    

## Display A and B Values

Generate and display the transfer matrix elements A and B for the 5-lens system.

In [39]:
# Generate measurements for the 5-lens system
measurements = generate_measurements(
    D_TRUE_FIT, 
    I0_TRUE_FIT, 
    CF_TRUE_FIT, 
    K_ROT, 
    wobble_grid_in, 
    defocus_grid_in
)

# Display summary
print("="*70)
print(f"A and B Values for {N_LENSES_FIT}-Lens System")
print("="*70)
print(f"\nTotal measurements: {len(measurements['A'])}")
print(f"Wobble range: {wobble_grid_in.min():.3f} to {wobble_grid_in.max():.3f}")
print(f"Defocus range: {defocus_grid_in.min()*1e3:.2f} to {defocus_grid_in.max()*1e3:.2f} mm")

# Display detailed measurements
print(f"\n{'Lens':>5s} {'Wobble':>8s} {'Defocus':>10s} {'A':>12s} {'B (m)':>12s} {'ψ (deg)':>10s}")
print("-"*70)

for i in range(len(measurements['A'])):
    lens_idx = int(measurements['wobble_lens'][i])
    wobble = float(measurements['wobble'][i])
    defocus = float(measurements['defocus'][i])
    A = float(measurements['A'][i])
    B = float(measurements['B'][i])
    psi_deg = float(np.degrees(measurements['psi'][i]))
    
    print(f"{lens_idx+1:>5d} {wobble:>8.3f} {defocus*1e3:>9.2f} {A:>12.6f} {B:>12.6f} {psi_deg:>10.2f}")

# Display statistics
print("\n" + "="*70)
print("Statistics:")
print(f"  A range:   [{measurements['A'].min():.6f}, {measurements['A'].max():.6f}]")
print(f"  B range:   [{measurements['B'].min():.6f}, {measurements['B'].max():.6f}] m")
print(f"  ψ range:   [{np.degrees(measurements['psi'].min()):.2f}, {np.degrees(measurements['psi'].max()):.2f}] deg")

A and B Values for 5-Lens System

Total measurements: 45
Wobble range: -0.100 to 0.100
Defocus range: 0.00 to 0.00 mm

 Lens   Wobble    Defocus            A        B (m)    ψ (deg)
----------------------------------------------------------------------
    1   -0.100      0.00 -18285.517945    -6.588222     445.30
    1   -0.075      0.00 -19603.272348    -9.223731     447.89
    1   -0.050      0.00 -20957.129611   -11.931446     450.47
    1   -0.025      0.00 -22347.089735   -14.711366     453.05
    1    0.000      0.00 -23773.152719   -17.563492     455.63
    1    0.025      0.00 -25235.318563   -20.487824     458.21
    1    0.050      0.00 -26733.587268   -23.484361     460.79
    1    0.075      0.00 -28267.958833   -26.553104     463.38
    1    0.100      0.00 -29838.433258   -29.694053     465.96
    2   -0.100      0.00 -16078.704740   -11.353286     445.91
    2   -0.075      0.00 -17926.384682   -12.844552     448.34
    2   -0.050      0.00 -19824.685993   -14.376676   

## Summary: Glaser Bell Model Parametrisation

### Physics

Each lens is characterised by two fitted parameters:
- **$I_{0,i}$** — nominal excitation (ampere-turns)
- **$C_{f,i}$** — geometry constant (bore, gap, pole-piece shape), unique to each lens

The accelerating voltage provides the **known** constant $K_v = e\mu_0/(2m_e v)$:
- **Rotation** $\psi_i = K_v \cdot I_{0,i}(1+w)$ → directly constrains $I_{0,i}$
- **Focal length** $1/f_i = C_{f,i} \cdot I_{0,i}^2(1+w)^2$ → uniquely determines $C_{f,i}$

### No degeneracy

Unlike a $(f_0, k_\phi)$ parametrisation, knowing $K_v$ means the rotation measurement independently constrains the excitation current, breaking the $f \leftrightarrow \psi$ degeneracy.

### Parameter count

| System | Distances | $I_0$ | $C_f$ | Total |
|--------|-----------|-------|--------|-------|
| 2-lens | 3 | 2 | 2 | **7** |
| 3-lens | 4 | 3 | 3 | **10** |
| N-lens | N+1 | N | N | **3N+1** |

In [40]:
# Print B/A values for all measurements
print("="*70)
print(f"B/A Ratio for {N_LENSES_FIT}-Lens System")
print("="*70)
print(f"\n{'Lens':>5s} {'Wobble':>8s} {'Defocus':>10s} {'A':>12s} {'B (m)':>12s} {'B/A':>12s}")
print("-"*70)

for i in range(len(measurements['A'])):
    lens_idx = int(measurements['wobble_lens'][i])
    wobble = float(measurements['wobble'][i])
    defocus = float(measurements['defocus'][i])
    A = float(measurements['A'][i])
    B = float(measurements['B'][i])
    B_over_A = B / A if A != 0 else np.inf
    
    print(f"{lens_idx+1:>5d} {wobble:>8.3f} {defocus*1e3:>9.2f} {A:>12.6f} {B:>12.6f} {B_over_A:>12.6e}")

# Summary statistics
print("\n" + "="*70)
print("B/A Statistics:")
B_over_A_all = measurements['B'] / measurements['A']
print(f"  B/A range:   [{B_over_A_all.min():.6e}, {B_over_A_all.max():.6e}]")
print(f"  B/A mean:    {B_over_A_all.mean():.6e}")
print(f"  B/A std:     {B_over_A_all.std():.6e}")

B/A Ratio for 5-Lens System

 Lens   Wobble    Defocus            A        B (m)          B/A
----------------------------------------------------------------------
    1   -0.100      0.00 -18285.517945    -6.588222 3.602973e-04
    1   -0.075      0.00 -19603.272348    -9.223731 4.705200e-04
    1   -0.050      0.00 -20957.129611   -11.931446 5.693263e-04
    1   -0.025      0.00 -22347.089735   -14.711366 6.583124e-04
    1    0.000      0.00 -23773.152719   -17.563492 7.387952e-04
    1    0.025      0.00 -25235.318563   -20.487824 8.118710e-04
    1    0.050      0.00 -26733.587268   -23.484361 8.784590e-04
    1    0.075      0.00 -28267.958833   -26.553104 9.393357e-04
    1    0.100      0.00 -29838.433258   -29.694053 9.951613e-04
    2   -0.100      0.00 -16078.704740   -11.353286 7.061070e-04
    2   -0.075      0.00 -17926.384682   -12.844552 7.165166e-04
    2   -0.050      0.00 -19824.685993   -14.376676 7.251906e-04
    2   -0.025      0.00 -21773.608672   -15.949656 7.3